# Socratic Tutor — LoRA Fine-tune with Unsloth

Fine-tunes **LLaMA 3.1 8B** (4-bit quantized) with LoRA on the `dataset.jsonl` generated in Stage 3.  
Target behavior: Socratic tutor for C programming, in Spanish.

**Run this notebook on a GPU instance** (Colab A100 / RunPod / Lambda Labs recommended).

---

## Prerequisites
```
pip install unsloth
```
Upload `data/dataset.jsonl` to the runtime (or load from HuggingFace Hub).

## 1 — Install & Imports

In [ ]:
# Install Unsloth (run once)
# !pip install unsloth
# !pip install --upgrade --no-cache-dir 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'

from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
import torch

print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2 — Load Base Model (4-bit)

In [ ]:
MAX_SEQ_LENGTH = 2048  # LLaMA 3.1 supports up to 128k; 2048 is enough for our dialogues
DTYPE          = None  # Auto-detect: bfloat16 on Ampere+, float16 elsewhere
LOAD_IN_4BIT   = True  # QLoRA — reduces VRAM from ~16 GB to ~5 GB

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = 'unsloth/Meta-Llama-3.1-8B-Instruct',
    max_seq_length  = MAX_SEQ_LENGTH,
    dtype           = DTYPE,
    load_in_4bit    = LOAD_IN_4BIT,
)

# Apply LLaMA 3.1 chat template so special tokens are set correctly
tokenizer = get_chat_template(tokenizer, chat_template='llama-3.1')

print('Model loaded ✓')

## 3 — Attach LoRA Adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r                   = 16,    # LoRA rank — 16 is a good balance for dialogue tasks
    target_modules      = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                            'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha          = 16,    # scaling factor — keeping it equal to r is standard
    lora_dropout        = 0,     # 0 is optimal for Unsloth
    bias                = 'none',
    use_gradient_checkpointing = 'unsloth',  # Unsloth's optimized checkpointing
    random_state        = 42,
    use_rslora          = False,  # set True to experiment with rank-stabilized LoRA
    loftq_config        = None,
)

model.print_trainable_parameters()

## 4 — Load & Format Dataset

In [ ]:
# ── Load JSONL ────────────────────────────────────────────────────────────────
# If running locally:
DATASET_PATH = 'data/dataset.jsonl'
# If uploaded to Colab:
# DATASET_PATH = '/content/dataset.jsonl'

raw_dataset = load_dataset('json', data_files=DATASET_PATH, split='train')
print(f'Total examples: {len(raw_dataset)}')
print(f'Columns: {raw_dataset.column_names}')

# ── Preview first example ─────────────────────────────────────────────────────
example = raw_dataset[0]
print(f"\nFirst conversation has {len(example['conversations'])} turns")
for turn in example['conversations'][:3]:
    print(f"  [{turn['role']}]: {turn['content'][:80]}...")

In [ ]:
# ── Apply chat template ────────────────────────────────────────────────────────
# Converts each conversation list into a single formatted string with
# <|begin_of_text|> / <|start_header_id|> ... <|eot_id|> tokens.
# We ONLY train on assistant turns (train_on_responses_only = True below).

def formatting_func(examples):
    texts = []
    for conversation in examples['conversations']:
        text = tokenizer.apply_chat_template(
            conversation,
            tokenize        = False,
            add_generation_prompt = False,
        )
        texts.append(text)
    return {'text': texts}

dataset = raw_dataset.map(formatting_func, batched=True)

print('Sample formatted text (first 400 chars):')
print(dataset[0]['text'][:400])

## 5 — Train

In [ ]:
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model        = model,
    tokenizer    = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field   = 'text',
        max_seq_length       = MAX_SEQ_LENGTH,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,   # effective batch = 8
        warmup_steps         = 10,
        num_train_epochs     = 3,          # start with 3; increase if loss plateaus
        learning_rate        = 2e-4,
        fp16                 = not torch.cuda.is_bf16_supported(),
        bf16                 = torch.cuda.is_bf16_supported(),
        logging_steps        = 5,
        optim                = 'adamw_8bit',
        weight_decay         = 0.01,
        lr_scheduler_type    = 'cosine',
        seed                 = 42,
        output_dir           = 'outputs/checkpoints',
        report_to            = 'none',     # set 'wandb' if you want logging
    ),
)

# Only compute loss on assistant turns — ignores user/system tokens
trainer = train_on_responses_only(
    trainer,
    instruction_part = '<|start_header_id|>user<|end_header_id|>\n\n',
    response_part    = '<|start_header_id|>assistant<|end_header_id|>\n\n',
)

print('Trainer ready ✓')

# ── Check memory before training ──────────────────────────────────────────────
if torch.cuda.is_available():
    gpu_stats  = torch.cuda.get_device_properties(0)
    start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
    max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
    print(f'GPU: {gpu_stats.name}  |  VRAM total: {max_memory} GB  |  reserved: {start_gpu_memory} GB')

In [ ]:
# ── Run training ──────────────────────────────────────────────────────────────
trainer_stats = trainer.train()

if torch.cuda.is_available():
    used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
    used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
    print(f'\nPeak VRAM used: {used_memory} GB  ({used_memory_for_lora} GB for LoRA)')

## 6 — Inference Test

In [ ]:
FastLanguageModel.for_inference(model)  # enable 2x faster inference

test_messages = [
    {
        'role': 'system',
        'content': 'Eres un tutor socrático de programación en lenguaje C. Tu objetivo no es dar respuestas directas, sino guiar al estudiante mediante preguntas que lo lleven a descubrir el concepto por sí mismo. Usa ejemplos del mundo real antes de introducir código. Valida cada respuesta del estudiante antes de avanzar.'
    },
    {
        'role': 'user',
        'content': 'Profe, ¿cómo funciona un bucle for en C?'
    }
]

inputs = tokenizer.apply_chat_template(
    test_messages,
    tokenize              = True,
    add_generation_prompt = True,
    return_tensors        = 'pt',
).to('cuda')

outputs = model.generate(
    input_ids        = inputs,
    max_new_tokens   = 512,
    temperature      = 0.7,
    top_p            = 0.9,
    repetition_penalty = 1.1,
    do_sample        = True,
)

response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
print('=== Model response ===')
print(response)

## 7 — Save Model

In [ ]:
SAVE_PATH = 'outputs/socratic-tutor-lora'

# Save LoRA adapters only (small — a few hundred MB)
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f'LoRA adapters saved → {SAVE_PATH}')

# ── Optional: merge + save full model in float16 ──────────────────────────────
# model.save_pretrained_merged(
#     'outputs/socratic-tutor-merged',
#     tokenizer,
#     save_method = 'merged_16bit',
# )

# ── Optional: save as GGUF (llama.cpp / Ollama) ───────────────────────────────
# model.save_pretrained_gguf(
#     'outputs/socratic-tutor-gguf',
#     tokenizer,
#     quantization_method = 'q4_k_m',  # good size/quality tradeoff
# )

## 8 — Push to HuggingFace Hub (optional)

```python
# model.push_to_hub('your-hf-username/socratic-tutor-llama3.1-8b-lora')
# tokenizer.push_to_hub('your-hf-username/socratic-tutor-llama3.1-8b-lora')
```

---
## Notes

| Hyperparameter | Value | Rationale |
|---|---|---|
| `r` (LoRA rank) | 16 | Good balance for dialogue style learning; increase to 32 if underfitting |
| `lora_alpha` | 16 | Equal to r — standard starting point |
| `num_train_epochs` | 3 | ~88 examples × 3 epochs = 264 steps; monitor loss |
| `learning_rate` | 2e-4 | Unsloth default; lower to 1e-4 if loss is unstable |
| `batch_size` | 2 × 4 grad_accum = 8 | Fits A100 40GB comfortably |
| `train_on_responses_only` | True | We only want to teach the assistant voice, not the student |

### Evaluation checklist (manual)
- [ ] Does the model open with a real-world analogy before any code?
- [ ] Does it avoid giving direct answers — guides with questions instead?
- [ ] Does it use the right register? (Nítido!, Exactoo, informal tutoring tone)
- [ ] Does it close with a practice exercise for the student?
- [ ] Does it handle wrong student answers gracefully (redirect, not reveal)?